# README
This notebook was used to explore the varying execution flows of worksessions. In the end this analysis was not included in the comparison module, but proved insightful for possible furhter analysis. 

Before exploring make sure to update the data columns and file and folder paths. 

In [ ]:
# import pandas as pd
import os
DT_FORMAT = "%d %b %Y %H:%M:%S,%f"
RUN_NAME = ''

graphs_output_paths = [
  f''
]

input_event_log_folder_path = f''

VERSION=14

In [ ]:
endtime_file_paths = [
]

import numpy as np
import pandas as pd
from collections import defaultdict
import matplotlib.pyplot as plt
import re

# ---------- CONFIG ----------
for path in graphs_output_paths:
    os.makedirs(path, exist_ok=True)

all_summaries = []
for path in endtime_file_paths:
    session_counts = []
    session_durations = []  # durations in minutes

    # ---------- LOAD DATA ----------
    df = pd.read_csv(path)
    df["source_file"] = path.split('\\')[-1].replace('.csv', '').replace('_V2_annotated','')

    # Ensure proper ordering
    df['Datetime'] = pd.to_datetime(df['Datetime'], format=DT_FORMAT, dayfirst=True)
    df = df.sort_values(['Study ID', 'Datetime'])

    # ---------- HELPERS ----------
    def ordinal(n):
        return ["first", "second", "third", "fourth", "fifth",
                "sixth", "seventh", "eighth", "ninth", "tenth"][n-1] if n <= 10 else f"{n}th"

    # ---------- PROCESS ----------
    worksessions = {}

    for study_id, group in df.groupby('Study ID'):
        group = group.sort_values('Datetime')
        
        session_count = 0
        current_sequence = []
        in_session = False
        session_start_time = None
        
        for _, row in group.iterrows():
            activity = row['Activity']
            time = row['Datetime']
            
            if activity == "1_Startup":
                current_sequence = ["1_Startup"]
                in_session = True
                session_count += 1
                session_start_time = time
            
            elif in_session:
                current_sequence.append(activity)
                
                if activity in ["7_Shutdown", "0B_Abrupt_End_Error"]:
                    session_end_time = time
                    
                    # ✅ Calculate duration (minutes)
                    if session_start_time is not None:
                        duration = (session_end_time - session_start_time).total_seconds() / 60
                        session_durations.append(duration)

                    key = "-".join(current_sequence)
                    
                    if key not in worksessions:
                        worksessions[key] = {
                            "occurence": 0,
                            "worksession_number": defaultdict(int)
                        }
                    
                    worksessions[key]["occurence"] += 1
                    worksessions[key]["worksession_number"][ordinal(session_count)] += 1
                    
                    in_session = False
                    current_sequence = []
                    session_start_time = None

        # ✅ store session count per Study ID
        session_counts.append(session_count)
    # Convert defaultdicts to normal dicts for clean output
    for v in worksessions.values():
        v["worksession_number"] = dict(v["worksession_number"])

    # ---------- RESULT ----------
    # Sort variants by occurrence
    sorted_variants = sorted(
        worksessions.items(),
        key=lambda x: x[1]['occurence'],
        reverse=True
    )

    labels = [k for k, _ in sorted_variants]
    values = [v['occurence'] for _, v in sorted_variants]

    plt.figure(figsize=(10, 6))
    plt.barh(labels, values)
    plt.xlabel("Frequency")
    plt.title(f"Worksession variant frequencies - {df['source_file'].iloc[0]}")
    plt.gca().invert_yaxis()  # highest on top
    plt.show()

    def clean_activity(activity):
        # Remove numeric prefix like "1_Startup" → "Startup"
        return re.sub(r'^\d+_?', '', activity)

    def clean_sequence(seq):
        return " → ".join(clean_activity(a) for a in seq.split("-"))

    rows = []

    # Collect all order labels across variants
    all_orders = set()

    for data in worksessions.values():
        all_orders.update(data["worksession_number"].keys())

    # Sort orders numerically (first, second, ...)

    def order_key(order):
        mapping = {
            "first": 1, "second": 2, "third": 3, "fourth": 4,
            "fifth": 5, "sixth": 6, "seventh": 7, "eighth": 8,
            "ninth": 9, "tenth": 10,
            "eleventh": 11, "twelfth": 12, "thirteenth": 13,
            "fourteenth": 14, "fifteenth": 15, "sixteenth": 16,
            "seventeenth": 17, "eighteenth": 18, "nineteenth": 19,
            "twentieth": 20
        }

        if order in mapping:
            return mapping[order]
        return int(''.join(filter(str.isdigit, order)))

    sorted_orders = sorted(all_orders, key=order_key)

    rows = []

    for variant, data in worksessions.items():
        sequence_clean = clean_sequence(variant)
        occ = data["occurence"]
        
        # Create a list of counts per order column
        order_counts = [
            data["worksession_number"].get(order, 0)
            for order in sorted_orders
        ]
        
        rows.append((sequence_clean, occ, order_counts))

    rows = sorted(rows, key=lambda x: x[1], reverse=True)
    # ---------- BUILD LATEX ----------
    latex = []
    latex.append("\\begin{table}[h!]")
    latex.append("\\centering")

    # Column format: sequence + count + N order columns
    col_format = "p{7cm} c " + " ".join(["c"] * len(sorted_orders))
    latex.append(f"\\begin{{tabular}}{{{col_format}}}")

    latex.append("\\hline")

    # Header row
    header_orders = " & ".join(sorted_orders)
    latex.append(f"\\textbf{{Worksession Sequence}} & \\textbf{{Count}} & {header_orders} \\\\")

    latex.append("\\hline")

    # Data rows
    for seq, occ, order_counts in rows:
        order_str = " & ".join(str(x) for x in order_counts)
        latex.append(f"{seq} & {occ} & {order_str} \\\\")

    latex.append("\\hline")
    latex.append("\\end{tabular}")
    latex.append(f"\\caption{{Worksession Variants and Session Order Distribution for {df['source_file'].iloc[0]}}}")
    latex.append("\\end{table}")

    latex_output = "\n".join(latex)

    print(latex_output)


    # ---------- SUMMARY STATISTICS ----------
    summary_data = {
        "Dataset": df['source_file'].iloc[0],
        "Avg Sessions": np.mean(session_counts) if session_counts else 0,
        "Median Sessions": np.median(session_counts) if session_counts else 0,
        "Min Sessions": np.min(session_counts) if session_counts else 0,
        "Max Sessions": np.max(session_counts) if session_counts else 0,
        "Avg Duration": np.mean(session_durations) if session_durations else 0,
        "Median Duration": np.median(session_durations) if session_durations else 0
    }

    summary_df = pd.DataFrame([summary_data])
    all_summaries.append(summary_data)

    print("\nSummary Statistics:")
    print(summary_df)

    for path in graphs_output_paths:
        plt.savefig(
            os.path.join(
                path,
                f"worksession_frequencies-{df['source_file'].iloc[0]}_V{VERSION}.png"
            ),
            dpi=300,
            bbox_inches="tight"
        )

# ---------- COMBINED LATEX TABLE ----------
latex_combined = []
latex_combined.append("\\begin{table}[h!]")
latex_combined.append("\\centering")

latex_combined.append("\\begin{tabular}{lcccccc}")
latex_combined.append("\\hline")

latex_combined.append(
    "Dataset & Avg Sessions & Median Sessions & Min Sessions & Max Sessions & Avg Duration (min) & Median Duration (min) \\\\"
)

latex_combined.append("\\hline")

for summary in all_summaries:
    latex_combined.append(
        f"{summary['Dataset']} & "
        f"{summary['Avg Sessions']:.2f} & "
        f"{summary['Median Sessions']:.2f} & "
        f"{summary['Min Sessions']:.0f} & "
        f"{summary['Max Sessions']:.0f} & "
        f"{summary['Avg Duration']:.2f} & "
        f"{summary['Median Duration']:.2f} \\\\"
    )

latex_combined.append("\\hline")
latex_combined.append("\\end{tabular}")
latex_combined.append("\\caption{Summary of worksession statistics across datasets}")
latex_combined.append("\\end{table}")

print("\n".join(latex_combined))
